In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path('..') if Path('.').name == 'notebooks' else Path('.')
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / 'master_df.csv', parse_dates=['ds'])
df = df.sort_values('ds').reset_index(drop=True)
print(df.shape)
print(df.dtypes)
print(df.head(3))
assert df['y'].isna().sum() == 0

df['price_mom_1m'] = df['y'].pct_change(periods=1)
df['price_mom_3m'] = df['y'].pct_change(periods=3)
df['price_accel'] = df['price_mom_1m'] - df['price_mom_3m']
df[['price_mom_1m', 'price_mom_3m', 'price_accel']] = df[['price_mom_1m', 'price_mom_3m', 'price_accel']].fillna(0)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(df['ds'], df['y'], label='Price (y)', color='tab:blue')
axes[0].fill_between(df['ds'], df['y'], alpha=0.1, color='tab:blue')
for _, row in df[df['harvest_window'] == 1].iterrows():
    axes[0].axvspan(row['ds'], row['ds'] + pd.offsets.MonthEnd(0), color='green', alpha=0.08)
axes[0].set_title('Raw price y with harvest window shading')
axes[0].legend()
axes[1].plot(df['ds'], df['price_mom_3m'], label='price_mom_3m', color='tab:orange')
axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_title('3-month price momentum')
axes[1].legend()
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'eda_price_momentum.png', bbox_inches='tight')
plt.close(fig)

feature_cols = [
    'price_mom_1m', 'price_mom_3m', 'price_accel',
    'production_dev_pct', 'harvest_window', 'lean_season',
    'harvest_proximity_days', 'rainfall_dev_pct'
]
corr = df[feature_cols + ['y']].corr()['y'].sort_values(ascending=False)
print('Feature correlations with price (y):')
print(corr.round(3))
for feature, value in corr.items():
    if feature != 'y' and abs(value) > 0.8:
        print(f'WARNING: high correlation with y detected for {feature}: {value:.3f}')

from statsmodels.tsa.seasonal import seasonal_decompose
decomp = seasonal_decompose(df.set_index('ds')['y'], model='multiplicative', period=12, extrapolate_trend='freq')
fig = decomp.plot()
fig.set_size_inches(10, 8)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'eda_seasonal_decomp.png', bbox_inches='tight')
plt.close(fig)

from statsmodels.tsa.stattools import adfuller
result = adfuller(df['y'].dropna())
print(f'ADF Statistic: {result[0]:.4f}')
print(f'p-value: {result[1]:.4f}')
print(f'Series is {if result[1] < 0.05 else }')

fig, ax = plt.subplots(figsize=(10, 5))
scatter = ax.scatter(
    df['production_dev_pct'],
    df['price_mom_3m'],
    c=df['harvest_window'],
    cmap='RdYlGn_r',
    alpha=0.7,
    s=60
)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('Production deviation from seasonal mean (supply signal)')
ax.set_ylabel('3-month price momentum')
ax.set_title('Supply vs Price Momentum — East Java Rice (Green = harvest window, Red = lean season)')
plt.colorbar(scatter, label='harvest_window')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'eda_supply_vs_momentum.png', bbox_inches='tight')
plt.close(fig)

df.to_csv(PROCESSED_DIR / 'master_df.csv', index=False)
print('Saved enriched master_df.csv')
print(f'Final columns: {df.columns.tolist()}')
print(f'Final shape: {df.shape}')

train_df = df[df['ds'] < '2024-01-01']
holdout_df = df[df['ds'] >= '2024-01-01']
train_df.to_csv(PROCESSED_DIR / 'train_df.csv', index=False)
holdout_df.to_csv(PROCESSED_DIR / 'holdout_df.csv', index=False)
print(f'Train rows:   {len(train_df)}')
print(f'Holdout rows: {len(holdout_df)}')